# Preprocessing et Feature Engineering

Notebook 2 : Nettoyage, création de features et sauvegarde des données prêtes pour la modélisation

## 1. CHARGEMENT DES DONNÉES

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Chargement du fichier nettoyé
df = pd.read_csv('data/silver/dvf_75_nettoye.csv', low_memory=False)

print("="*70)
print("CHARGEMENT DES DONNÉES")
print("="*70)
print(f"\n[OK] Données chargées : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nTaille mémoire : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nTypes de données :")
print(df.dtypes)
print(f"\nAperçu des premières lignes :")
df.head()

CHARGEMENT DES DONNÉES

[OK] Données chargées : 204,125 lignes x 41 colonnes

Taille mémoire : 200.2 MB

Types de données :
id_mutation                         str
date_mutation                       str
numero_disposition                int64
nature_mutation                     str
valeur_fonciere                 float64
adresse_numero                  float64
adresse_suffixe                     str
adresse_nom_voie                    str
adresse_code_voie                   str
code_postal                     float64
code_commune                      int64
nom_commune                         str
code_departement                  int64
ancien_code_commune             float64
ancien_nom_commune              float64
id_parcelle                         str
ancien_id_parcelle              float64
numero_volume                   float64
lot1_numero                         str
lot1_surface_carrez             float64
lot2_numero                         str
lot2_surface_carrez             floa

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,prix_m2
0,2020-814508,2020-07-01,1,Vente,148000.0,105.0,NaN,RUE VIEILLE DU TEMPLE,9783,75003.0,...,12.0,1.0,NaN,NaN,NaN,NaN,NaN,2.361559,48.860524,12333.333333
1,2020-814510,2020-07-01,1,Vente,270000.0,9.0,NaN,RUE HEROLD,4619,75001.0,...,27.0,2.0,NaN,NaN,NaN,NaN,NaN,2.341301,48.864895,10000.000000
2,2020-814514,2020-07-06,1,Vente,2603550.0,3.0,NaN,RUE DE L ECHELLE,3082,75001.0,...,137.0,6.0,NaN,NaN,NaN,NaN,NaN,2.333735,48.863683,19004.014599
3,2020-814515,2020-07-02,1,Vente,1112000.0,98.0,NaN,RUE SAINT DENIS,8525,75001.0,...,84.0,4.0,NaN,NaN,NaN,NaN,NaN,2.350170,48.863265,13238.095238
4,2020-814517,2020-07-02,1,Vente,1670000.0,20.0,NaN,RUE DES BONS ENFANTS,1105,75001.0,...,120.0,5.0,NaN,NaN,NaN,NaN,NaN,2.338875,48.863430,13916.666667


In [2]:
# Vérifier les colonnes disponibles
print("\nColonnes disponibles :")
print(df.columns.tolist())

# Vérifier les valeurs manquantes
print("\n" + "="*70)
print("VALEURS MANQUANTES (TOP 10)")
print("="*70)
missing = df.isnull().sum().sort_values(ascending=False).head(10)
for col, count in missing.items():
    pct = count / len(df) * 100
    print(f"  {col:30} : {count:6,} ({pct:5.1f}%)")


Colonnes disponibles :
['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude', 'prix_m2']

VALEURS MANQUANTES (TOP 10)
  ancien_nom_commune             : 204,125 (100.0%)
  numero_volume                  : 204,125 (100.0%)
  ancien_code_commune            : 204,125 (100.0%)
  ancien_i

## 2. FILTRAGE IMPORTANT

In [3]:
print("\n" + "="*70)
print("ÉTAPE 1 : FILTRAGE DES DONNÉES")
print("="*70)

df_filtered = df.copy()
initial_rows = len(df_filtered)

# 1. Garder seulement les ventes normales
if 'nature_mutation' in df_filtered.columns:
    df_filtered = df_filtered[df_filtered['nature_mutation'] == 'Vente']
    rows_after_nature = len(df_filtered)
    removed = initial_rows - rows_after_nature
    print(f"\n[FILTER 1] Nature mutation = 'Vente'")
    print(f"  Avant : {initial_rows:,} lignes")
    print(f"  Après : {rows_after_nature:,} lignes (- {removed:,} lignes)")

else:
    print(f"\n[WARNING] Colonne 'nature_mutation' non trouvée")


ÉTAPE 1 : FILTRAGE DES DONNÉES

[FILTER 1] Nature mutation = 'Vente'
  Avant : 204,125 lignes
  Après : 201,516 lignes (- 2,609 lignes)


In [4]:
# 2. Garder seulement les appartements
if 'type_local' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['type_local'] == 'Appartement']
    after = len(df_filtered)
    removed = before - after
    print(f"\n[FILTER 2] Type local = 'Appartement'")
    print(f"  Avant : {before:,} lignes")
    print(f"  Après : {after:,} lignes (- {removed:,} lignes)")

else:
    print(f"\n[WARNING] Colonne 'type_local' non trouvée")


[FILTER 2] Type local = 'Appartement'
  Avant : 201,516 lignes
  Après : 176,151 lignes (- 25,365 lignes)


In [5]:
# 3. Garder seulement les arrondissements de Paris (75 + 2 chiffres = 75001-75020)
if 'code_postal' in df_filtered.columns:
    before = len(df_filtered)
    # Convertir en string et filtrer les codes postaux 75xxx
    df_filtered['code_postal'] = df_filtered['code_postal'].astype(str)
    df_filtered = df_filtered[df_filtered['code_postal'].str.startswith('75')]
    after = len(df_filtered)
    removed = before - after
    print(f"\n[FILTER 3] Code postal = 75xxx (Paris)")
    print(f"  Avant : {before:,} lignes")
    print(f"  Après : {after:,} lignes (- {removed:,} lignes)")

else:
    print(f"\n[WARNING] Colonne 'code_postal' non trouvée")


[FILTER 3] Code postal = 75xxx (Paris)
  Avant : 176,151 lignes
  Après : 176,149 lignes (- 2 lignes)


In [6]:
# 4. Supprimer les outliers sur prix_m2
# Calculer prix_m2 si non présent
if 'prix_m2' not in df_filtered.columns:
    df_filtered['prix_m2'] = df_filtered['valeur_fonciere'] / df_filtered['surface_reelle_bati']

before = len(df_filtered)

# Définir les limites (500 EUR/m² à 30 000 EUR/m²)
q1 = df_filtered['prix_m2'].quantile(0.01)
q99 = df_filtered['prix_m2'].quantile(0.99)
lower_bound = max(500, q1)
upper_bound = min(30000, q99)

df_filtered = df_filtered[(df_filtered['prix_m2'] >= lower_bound) & 
                           (df_filtered['prix_m2'] <= upper_bound)]
after = len(df_filtered)
removed = before - after

print(f"\n[FILTER 4] Supprimer outliers prix_m2")
print(f"  Limites : [{lower_bound:,.0f} EUR/m² ; {upper_bound:,.0f} EUR/m²]")
print(f"  Avant : {before:,} lignes")
print(f"  Après : {after:,} lignes (- {removed:,} lignes)")

print(f"\n" + "="*70)
print(f"RÉSUMÉ FILTRAGE")
print(f"="*70)
print(f"Lignes initiales : {initial_rows:,}")
print(f"Lignes finales   : {len(df_filtered):,}")
print(f"Supprimées       : {initial_rows - len(df_filtered):,} ({(initial_rows - len(df_filtered))/initial_rows*100:.1f}%)")


[FILTER 4] Supprimer outliers prix_m2
  Limites : [500 EUR/m² ; 30,000 EUR/m²]
  Avant : 176,149 lignes
  Après : 152,419 lignes (- 23,730 lignes)

RÉSUMÉ FILTRAGE
Lignes initiales : 204,125
Lignes finales   : 152,419
Supprimées       : 51,706 (25.3%)


## 3. FEATURE ENGINEERING

In [7]:
print("\n" + "="*70)
print("ÉTAPE 2 : CRÉATION DE NOUVELLES FEATURES")
print("="*70)

# Créer une copie pour le feature engineering
df_processed = df_filtered.copy()

# 1. log_prix_m2
df_processed['log_prix_m2'] = np.log(df_processed['prix_m2'])
print(f"\n[FEATURE 1] log_prix_m2 = log(prix_m2)")
print(f"  Min  : {df_processed['log_prix_m2'].min():.4f}")
print(f"  Mean : {df_processed['log_prix_m2'].mean():.4f}")
print(f"  Max  : {df_processed['log_prix_m2'].max():.4f}")


ÉTAPE 2 : CRÉATION DE NOUVELLES FEATURES

[FEATURE 1] log_prix_m2 = log(prix_m2)
  Min  : 6.2146
  Mean : 9.2371
  Max  : 10.3090


In [8]:
# 2. surface_totale (surface_bati + surfaces carrez)
# Chercher toutes les colonnes de surface
surface_cols = [col for col in df_processed.columns if 'surface' in col.lower()]
print(f"\n[FEATURE 2] surface_totale")
print(f"  Colonnes de surface trouvées : {surface_cols}")

# Créer surface_totale
if 'surface_reelle_bati' in df_processed.columns:
    # Commencer avec la surface bâtie
    df_processed['surface_totale'] = df_processed['surface_reelle_bati'].fillna(0)
    
    # Ajouter les surfaces carrez s'il y en a
    carrez_cols = [col for col in surface_cols if 'carrez' in col.lower()]
    for col in carrez_cols:
        if col in df_processed.columns:
            df_processed['surface_totale'] += df_processed[col].fillna(0)
    
    print(f"  Colonnes utilisées : surface_reelle_bati + {carrez_cols}")
    print(f"  Min : {df_processed['surface_totale'].min():.1f} m²")
    print(f"  Mean : {df_processed['surface_totale'].mean():.1f} m²")
    print(f"  Max : {df_processed['surface_totale'].max():.1f} m²")


[FEATURE 2] surface_totale
  Colonnes de surface trouvées : ['lot1_surface_carrez', 'lot2_surface_carrez', 'lot3_surface_carrez', 'lot4_surface_carrez', 'lot5_surface_carrez', 'surface_reelle_bati', 'surface_terrain']
  Colonnes utilisées : surface_reelle_bati + ['lot1_surface_carrez', 'lot2_surface_carrez', 'lot3_surface_carrez', 'lot4_surface_carrez', 'lot5_surface_carrez']
  Min : 11.0 m²
  Mean : 94.1 m²
  Max : 14842.0 m²


In [9]:
# 3. Année et mois de la mutation
if 'date_mutation' in df_processed.columns:
    df_processed['date_mutation'] = pd.to_datetime(df_processed['date_mutation'], errors='coerce')
    df_processed['annee'] = df_processed['date_mutation'].dt.year
    df_processed['mois'] = df_processed['date_mutation'].dt.month
    
    print(f"\n[FEATURE 3] Extraction date_mutation")
    print(f"  annee : Min={df_processed['annee'].min()}, Max={df_processed['annee'].max()}")
    print(f"  mois  : Min={df_processed['mois'].min()}, Max={df_processed['mois'].max()}")

else:
    print(f"\n[WARNING] Colonne 'date_mutation' non trouvée")


[FEATURE 3] Extraction date_mutation
  annee : Min=2020, Max=2025
  mois  : Min=1, Max=12


In [10]:
# 4. Colonne arrondissement (1 à 20)
if 'arrondissement' in df_processed.columns:
    print(f"\n[FEATURE 4] arrondissement (colonne existante)")
    print(f"  Valeurs uniques : {sorted(df_processed['arrondissement'].unique())}")
elif 'code_postal' in df_processed.columns:
    # Extraire l'arrondissement du code postal (75001 -> 1, 75020 -> 20)
    df_processed['code_postal'] = df_processed['code_postal'].astype(str)
    df_processed['arrondissement'] = df_processed['code_postal'].str[2:4].astype(int)
    df_processed['arrondissement'] = df_processed['arrondissement'].clip(lower=1, upper=20)
    
    print(f"\n[FEATURE 4] arrondissement (extrait du code_postal)")
    print(f"  Valeurs uniques : {sorted(df_processed['arrondissement'].unique())}")
else:
    print(f"\n[WARNING] Impossible de créer l'arrondissement")


[FEATURE 4] arrondissement (extrait du code_postal)
  Valeurs uniques : [np.int64(1), np.int64(2)]


In [11]:
# 5. Distance approximative au centre de Paris (Île de la Cité: 48.8530, 2.3499)
if 'latitude' in df_processed.columns and 'longitude' in df_processed.columns:
    paris_center_lat = 48.8530
    paris_center_lon = 2.3499
    
    # Formule approximative (Haversine simplifiée en km)
    def haversine_distance(lat, lon, center_lat, center_lon):
        lat_rad = np.radians(lat - center_lat)
        lon_rad = np.radians(lon - center_lon)
        a = np.sin(lat_rad/2)**2 + np.cos(np.radians(center_lat)) * np.cos(np.radians(lat)) * np.sin(lon_rad/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return 6371 * c  # rayon terre en km
    
    # Gérer les NaN
    mask = df_processed['latitude'].notna() & df_processed['longitude'].notna()
    df_processed['distance_center_km'] = np.nan
    df_processed.loc[mask, 'distance_center_km'] = haversine_distance(
        df_processed.loc[mask, 'latitude'].values,
        df_processed.loc[mask, 'longitude'].values,
        paris_center_lat,
        paris_center_lon
    )
    
    print(f"\n[FEATURE 5] distance_center_km")
    print(f"  Min  : {df_processed['distance_center_km'].min():.2f} km")
    print(f"  Mean : {df_processed['distance_center_km'].mean():.2f} km")
    print(f"  Max  : {df_processed['distance_center_km'].max():.2f} km")
    print(f"  NaN  : {df_processed['distance_center_km'].isna().sum():,}")

else:
    print(f"\n[NOTE] Colonnes latitude/longitude non disponibles (distance non calculée)")


[FEATURE 5] distance_center_km
  Min  : 0.06 km
  Mean : 3.50 km
  Max  : 6.98 km
  NaN  : 280


## 4. SÉLECTION DES COLONNES FINALES

In [12]:
print("\n" + "="*70)
print("ÉTAPE 3 : SÉLECTION DES COLONNES FINALES")
print("="*70)

# Définir les colonnes à conserver
features_to_keep = [
    # Features numériques principales
    'surface_reelle_bati',
    'surface_terrain',
    'nombre_pieces_principales',
    'nombre_lots',
    'surface_totale',
    
    # Features temporelles
    'annee',
    'mois',
    
    # Features géographiques
    'arrondissement',
    'code_postal',
    
    # Features optionnelles
    'distance_center_km',
    'latitude',
    'longitude',
    
    # Features d'identification
    'type_local',
    'nature_mutation',
]

# Cible
target_col = 'log_prix_m2'

# Vérifier quelles colonnes existent
existing_features = [col for col in features_to_keep if col in df_processed.columns]
missing_features = [col for col in features_to_keep if col not in df_processed.columns]

print(f"\n[OK] Colonnes feature conservées : {len(existing_features)}/{len(features_to_keep)}")
print(f"     {existing_features}")

if missing_features:
    print(f"\n[WARNING] Colonnes manquantes : {missing_features}")

print(f"\n[OK] Colonne cible : {target_col}")


ÉTAPE 3 : SÉLECTION DES COLONNES FINALES

[OK] Colonnes feature conservées : 14/14
     ['surface_reelle_bati', 'surface_terrain', 'nombre_pieces_principales', 'nombre_lots', 'surface_totale', 'annee', 'mois', 'arrondissement', 'code_postal', 'distance_center_km', 'latitude', 'longitude', 'type_local', 'nature_mutation']

[OK] Colonne cible : log_prix_m2


In [13]:
# Toutes les colonnes à garder (features + cible)
final_columns = existing_features + [target_col]

# Créer le dataframe final
df_final = df_processed[final_columns].copy()

print(f"\n" + "="*70)
print("DATAFRAME FINAL")
print(f"="*70)
print(f"\nShape : {df_final.shape[0]:,} lignes x {df_final.shape[1]} colonnes")
print(f"\nValeurs manquantes :")
missing = df_final.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    for col, count in missing.items():
        pct = count / len(df_final) * 100
        print(f"  {col:30} : {count:6,} ({pct:5.1f}%)")
else:
    print("  Aucune valeur manquante!")

print(f"\nStatistiques descriptives :")
df_final.describe()


DATAFRAME FINAL

Shape : 152,419 lignes x 15 colonnes

Valeurs manquantes :
  surface_terrain                : 151,231 ( 99.2%)
  distance_center_km             :    280 (  0.2%)
  latitude                       :    280 (  0.2%)
  longitude                      :    280 (  0.2%)

Statistiques descriptives :


,surface_reelle_bati,surface_terrain,nombre_pieces_principales,nombre_lots,surface_totale,annee,mois,arrondissement,distance_center_km,latitude,longitude,log_prix_m2
count,152419.000000,1188.000000,152419.000000,152419.000000,152419.000000,152419.000000,152419.000000,152419.000000,152139.000000,152139.000000,152139.000000,152419.000000
mean,53.790754,381.862795,2.422893,1.676746,94.100285,2022.343304,6.597583,1.065674,3.500019,48.861951,2.340857,9.237077
std,39.232924,310.523932,1.283125,0.825540,85.423322,1.432317,3.416954,0.247713,1.314233,0.020007,0.037680,0.417128
min,11.000000,19.000000,0.000000,0.000000,11.000000,2020.000000,1.000000,1.000000,0.056608,48.818759,2.255896,6.214608
25%,28.000000,190.000000,1.000000,1.000000,46.000000,2021.000000,4.000000,1.000000,2.542705,48.845389,2.312400,9.088980
50%,43.000000,300.000000,2.000000,2.000000,73.000000,2022.000000,7.000000,1.000000,3.601017,48.862123,2.343625,9.262300
75%,68.000000,516.000000,3.000000,2.000000,119.105000,2023.000000,10.000000,1.000000,4.450872,48.879665,2.371768,9.430814
max,1500.000000,3262.000000,34.000000,31.000000,14842.000000,2025.000000,12.000000,2.000000,6.975001,48.900565,2.412825,10.308953


## 5. SAUVEGARDE DES DONNÉES TRAITÉES

In [14]:
print("\n" + "="*70)
print("ÉTAPE 4 : SAUVEGARDE DES DONNÉES")
print("="*70)

# Créer les X (features) et y (cible)
X = df_final.drop(columns=[target_col])
y = df_final[[target_col]]

print(f"\nX (Features) : {X.shape[0]:,} lignes x {X.shape[1]} colonnes")
print(f"y (Target)   : {y.shape[0]:,} lignes x {y.shape[1]} colonnes")


ÉTAPE 4 : SAUVEGARDE DES DONNÉES

X (Features) : 152,419 lignes x 14 colonnes
y (Target)   : 152,419 lignes x 1 colonnes


In [15]:
# Sauvegarder les fichiers
output_paths = {
    'complete': 'data/gold/model_data_processed.csv',
    'features': 'data/gold/X_features.csv',
    'target': 'data/gold/y_target.csv'
}

# Dataframe complet
df_final.to_csv(output_paths['complete'], index=False, encoding='utf-8')
print(f"\n[OK] Dataframe complet sauvegardé")
print(f"    {output_paths['complete']}")
print(f"    Taille : {df_final.shape[0]:,} x {df_final.shape[1]}")
print(f"    Fichier KB : {pd.read_csv(output_paths['complete']).memory_usage(deep=True).sum() / 1024:.1f} KB")


[OK] Dataframe complet sauvegardé
    data/gold/model_data_processed.csv
    Taille : 152,419 x 15
    Fichier KB : 32448.7 KB


In [16]:
# Features (X)
X.to_csv(output_paths['features'], index=False, encoding='utf-8')
print(f"\n[OK] Features (X) sauvegardées")
print(f"    {output_paths['features']}")
print(f"    Taille : {X.shape[0]:,} x {X.shape[1]}")

# Target (y)
y.to_csv(output_paths['target'], index=False, encoding='utf-8')
print(f"\n[OK] Target (y) sauvegardée")
print(f"    {output_paths['target']}")
print(f"    Taille : {y.shape[0]:,} x {y.shape[1]}")


[OK] Features (X) sauvegardées
    data/gold/X_features.csv
    Taille : 152,419 x 14

[OK] Target (y) sauvegardée
    data/gold/y_target.csv
    Taille : 152,419 x 1


In [17]:
# Afficher un résumé final
print("\n" + "="*70)
print("RÉSUMÉ FINAL")
print("="*70)

print(f"\n[OK] Preprocessing terminé avec succès!")
print(f"\nDonnées disponibles :")
print(f"  1. model_data_processed.csv      : Dataframe complet ({df_final.shape[0]:,} x {df_final.shape[1]})")
print(f"  2. X_features.csv                : Features uniquement ({X.shape[0]:,} x {X.shape[1]})")
print(f"  3. y_target.csv                  : Cible (log_prix_m2) ({y.shape[0]:,} x {y.shape[1]})")

print(f"\nColonnes finales ({X.shape[1]} features) :")
for i, col in enumerate(X.columns, 1):
    dtype = X[col].dtype
    non_null = X[col].notna().sum()
    print(f"  {i:2}. {col:30} ({dtype}) - {non_null:,} valeurs")

print(f"\nCible :")
print(f"  {target_col:30} (float64) - {y[target_col].notna().sum():,} valeurs")
print(f"  Min  : {y[target_col].min():.4f}")
print(f"  Mean : {y[target_col].mean():.4f}")
print(f"  Max  : {y[target_col].max():.4f}")


RÉSUMÉ FINAL

[OK] Preprocessing terminé avec succès!

Données disponibles :
  1. model_data_processed.csv      : Dataframe complet (152,419 x 15)
  2. X_features.csv                : Features uniquement (152,419 x 14)
  3. y_target.csv                  : Cible (log_prix_m2) (152,419 x 1)

Colonnes finales (14 features) :
   1. surface_reelle_bati            (float64) - 152,419 valeurs
   2. surface_terrain                (float64) - 1,188 valeurs
   3. nombre_pieces_principales      (float64) - 152,419 valeurs
   4. nombre_lots                    (int64) - 152,419 valeurs
   5. surface_totale                 (float64) - 152,419 valeurs
   6. annee                          (int32) - 152,419 valeurs
   7. mois                           (int32) - 152,419 valeurs
   8. arrondissement                 (int64) - 152,419 valeurs
   9. code_postal                    (str) - 152,419 valeurs
  10. distance_center_km             (float64) - 152,139 valeurs
  11. latitude                       (f